# Analysis of Symbolic Regression equations
A manual analysis of selected equations.

In [16]:
import numpy as np
import re as regex
import openml
import sympy as sp

from scipy.optimize import minimize
from sklearn.metrics import r2_score

def get_task_clean_data_and_name(task_id) :
    """
    Get the data and the name of the dataset and target for a given task ID.
    """
    task = openml.tasks.get_task(task_id, download_splits=True,
                                     download_data=True, download_qualities=True,
                                     download_features_meta_data=True)
        
    # the 'task' object above contains a lot of useful information,
    # like the name of the target variable and the id of the dataset
    df_X, df_y = task.get_X_and_y('dataframe')
    
    # check if there is any missing value
    # here below there is a sum().sum() because it is adding up missing values
    # in rows AND THEN in columns
    missing_data = df_X.isnull().sum().sum() + df_y.isnull().sum()
    
    if missing_data > 0 :
        # we actually have to go with a task/dataset-specific correction, I think,
        # as there are only two datasets with missing values
        if task_id == 361268 : # dataset fps_benchmark
            # this task has several columns with A LOT of missing data,
            # so we are just going to drop them
            df_X.dropna(axis=1, inplace=True)
        elif task_id == 361616 : # dataset Moneyball
            # again, a few columns with 800/1200 missing values, get dropped
            df_X.dropna(axis=1, inplace=True)
    
    # check if there are any categorical columns
    df_categorical = df_X.select_dtypes(include=['category', 'object'])
    categorical_features = df_categorical.shape[1]
    
    # convert categorical columns to numerical values
    for c in df_categorical.columns :
        df_X[c] = df_X[c].astype('category') # double-check that it is treated as a categorical column
        df_X[c] = df_X[c].cat.codes # replace values with category codes (automatically computed)
    
    X = df_X.values
    y = df_y.values
    
    # let's also get the name of the dataset
    dataset = task.get_dataset()
        
    return df_X, df_y, dataset, task, missing_data, categorical_features

## abalone

In [ ]:
task_id = 361234
personalized_equation = "-0.0323031301921013 + (0.000204087575289272*x1 - 0.000185365221986661*x4 + 1.54799300186112e-6*x6 + 7.11224454083269e-5*x7)/(2.95765583621071e-5*x5 + 3.24193098237202e-6)"
personalized_equation = "3.73667019003373 + (10611898646893.3*x1 + 7007239420672.58*x4 - 16363792324998.6*x6 + 23095740989.1673*x7)/(1325768473061.23*x5 + 353966817487.809)"
personalized_equation = "-0.0323031301921013 + (-0.000185365221986661*x1 + 7.11224454083269e-5*x4 + 1.54799300186112e-6*x6 + 0.000204087575289272*x7)/(2.95765583621071e-5*x5 + 3.24193098237202e-6)"
personalized_equation = "6.0091949922108 + (-0.000396651914382866*x1 + 0.136164661078217*x4 - 0.0322536530441093*x6 + 0.0602221058849336*x7)/(0.00544078034214355 - 0.000993963605486663*x5)"
personalized_equation = "-x1 + x4 - (x1 - 0.513) + (x3 + (x4 + x7 - (x5 + x6))/(x5+0.143)) * 5.90 + 3.01"


# parse equation, convert it first to symbolic representation, then to a lambdified executable expression
expr = sp.sympify(personalized_equation)
print("Personalized equation:", expr)
variables = sorted([s for s in expr.free_symbols if s.name.startswith('x')], key=lambda s: s.name)
print("Variables of the personalized equation:", variables)
features = [str(v) for v in variables]
lambdified_function = sp.lambdify(variables, expr, modules='numpy')

# load dataset
df_X, df_y, dataset, task, missing_data, categorical_features = get_task_clean_data_and_name(task_id)
df_X.columns = ["x%d" % i for i in range(1, len(df_X.columns)+1)]
print(df_X.columns)
X = df_X[features].values
y = df_y.values

y_pred = lambdified_function(*X.T)
r2_value = r2_score(y, y_pred)
print("R2 value of the lambdified expression: %.4f" % r2_value)

Personalized equation: -2*x1 + 5.9*x3 + x4 + 3.523 + 5.9*(x4 - x5 - x6 + x7)/(x5 + 0.143)
Variables of the personalized equation: [x1, x3, x4, x5, x6, x7]
Index(['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8'], dtype='object')
R2 value of the lambdified expression: -11.4366


## Moneyball

In [ ]:
task_id = 361616
personalized_equation = ""

# parse equation, convert it first to symbolic representation, then to a lambdified executable expression
expr = sp.sympify(personalized_equation)
print("Personalized equation:", expr)
variables = sorted([s for s in expr.free_symbols if s.name.startswith('x')], key=lambda s: s.name)
print("Variables of the personalized equation:", variables)
lambdified_function = sp.lambdify(variables, expr, modules='numpy')

# load dataset
df_X, df_y, dataset, task, missing_data, categorical_features = get_task_clean_data_and_name(task_id)
df_X.columns = ["x%d" % i for i in range(1, len(df_X.columns)+1)]
print(df_X.columns)
X = df_X[[str(v) for v in variables]].values
y = df_y.values

y_pred = lambdified_function(*X.T)
r2_value = r2_score(y, y_pred)
print("R2 value of the lambdified expression: %.4f" % r2_value)

Personalized equation: 0.617*x4 + x5*(-0.002*x3 + 27.636*x6) - 165.422
Variables of the personalized equation: [x3, x4, x5, x6]
Index(['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10'], dtype='object')
R2 value of the lambdified expression: 0.8009
